# Step 01_02_01 -- DuckDB Ingestion: aoe2companion

**Phase:** 01 -- Data Exploration
**Pipeline Section:** 01_02 -- EDA
**Dataset:** aoe2companion
**Question:** Can we ingest all aoe2companion Parquet and CSV files into
DuckDB, handling binary column annotations and CSV type inference?
**Invariants applied:** #6 (reproducibility), #9 (step scope)
**Step scope:** query

In [ ]:
import json
import logging
from pathlib import Path

from rts_predict.common.notebook_utils import get_reports_dir
from rts_predict.games.aoe2.config import (
    AOE2COMPANION_RAW_DIR,
    AOE2COMPANION_DB_FILE,
)
from rts_predict.games.aoe2.datasets.aoe2companion.pre_ingestion import (
    inspect_binary_columns,
    run_smoke_test,
    ingest_matches_raw,
    ingest_ratings_raw,
    ingest_leaderboards_raw,
    ingest_profiles_raw,
    verify_tables,
    check_ratings_types,
    count_won_nulls,
    find_date_gap,
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 1. Pyarrow binary column inspection

In [ ]:
binary_info = inspect_binary_columns(AOE2COMPANION_RAW_DIR)
for subdir, info in binary_info.items():
    print(f"{subdir}: {info['binary_column_count']} binary columns")
    for col in info["binary_columns"]:
        print(f"  {col['name']}: {col['converted_type']}")

## 2. Smoke test

In [ ]:
import duckdb

con_smoke = duckdb.connect(":memory:")
smoke = run_smoke_test(con_smoke, AOE2COMPANION_RAW_DIR)
print(f"Smoke matches: {smoke['matches']['row_count']} rows, {smoke['matches']['column_count']} cols")
print(f"Smoke ratings: {smoke['ratings']['row_count']} rows, {smoke['ratings']['column_count']} cols")
con_smoke.close()

## 3. Full ingestion

In [ ]:
import duckdb

AOE2COMPANION_DB_FILE.parent.mkdir(parents=True, exist_ok=True)
if AOE2COMPANION_DB_FILE.exists():
    AOE2COMPANION_DB_FILE.unlink()

con = duckdb.connect(str(AOE2COMPANION_DB_FILE))
con.execute("SET memory_limit = '24GB'")
con.execute("SET threads = 4")

m = ingest_matches_raw(con, AOE2COMPANION_RAW_DIR)
print(f"matches_raw: {m:,} rows")

In [ ]:
# ratings_raw: use explicit types (read_csv_auto fails on full 2072 files)
r = ingest_ratings_raw(con, AOE2COMPANION_RAW_DIR, use_explicit_types=True)
print(f"ratings_raw: {r:,} rows")

In [ ]:
l = ingest_leaderboards_raw(con, AOE2COMPANION_RAW_DIR)
print(f"leaderboards_raw: {l:,} rows")

p = ingest_profiles_raw(con, AOE2COMPANION_RAW_DIR)
print(f"profiles_raw: {p:,} rows")

## 4. Post-ingestion verification

In [ ]:
v = verify_tables(con)
for table, info in v.items():
    print(f"\n{table}: {info['row_count']:,} rows, {info['column_count']} columns")

In [ ]:
rt = check_ratings_types(con)
print("Ratings types:")
for col, typ in rt.items():
    print(f"  {col}: {typ}")

In [ ]:
wn = count_won_nulls(con)
print(f"won column: {wn['null_count']:,} NULL / {wn['total']:,} total")

In [ ]:
dg = find_date_gap(AOE2COMPANION_RAW_DIR)
print(f"Matches: {dg['matches_date_count']} dates")
print(f"Ratings: {dg['ratings_date_count']} dates")
print(f"In matches not ratings: {dg['in_matches_not_ratings']}")

In [ ]:
con.close()

## 5. Artifacts

In [ ]:
artifacts_dir = (
    get_reports_dir("aoe2", "aoe2companion")
    / "artifacts" / "01_exploration" / "01_eda"
)
print(f"Artifacts at: {artifacts_dir}")
print("JSON and MD artifacts written by pre_ingestion pipeline.")